# 🍬 Nassau Candy – Factory Reallocation & Shipping Optimization
Colab notebook: data prep → EDA → lead-time models → clustering → simulation → recommendations → Streamlit app.

**Run order:** run all cells top to bottom. Upload `Nassau_Candy_Distributor.csv` when asked (Colab only - the dashboard itself has no upload box).

In [ ]:
!pip -q install streamlit plotly scikit-learn pandas numpy
from google.colab import files
up = files.upload()   # choose Nassau_Candy_Distributor.csv (saved in /content, the dashboard reads it from there)


## 1. Shared pipeline module
Written to `core.py` so the notebook and the Streamlit app use identical logic.

In [ ]:
%%writefile core.py
"""Nassau Candy - Factory Reallocation & Shipping Optimization: core pipeline.
Shared by the Colab notebook and the Streamlit app."""
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans

FACTORIES = {
    "Lot's O' Nuts": (32.881893, -111.768036),
    "Wicked Choccy's": (32.076176, -81.088371),
    "Sugar Shack": (48.11914, -96.18115),
    "Secret Factory": (41.446333, -90.565487),
    "The Other Factory": (35.1175, -89.971107),
}
PRODUCT_FACTORY = {
    "Wonka Bar - Nutty Crunch Surprise": "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows": "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious": "Lot's O' Nuts",
    "Wonka Bar - Milk Chocolate": "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel": "Wicked Choccy's",
    "Laffy Taffy": "Sugar Shack", "SweeTARTS": "Sugar Shack", "Nerds": "Sugar Shack",
    "Fun Dip": "Sugar Shack", "Fizzy Lifting Drinks": "Sugar Shack",
    "Everlasting Gobstopper": "Secret Factory", "Lickable Wallpaper": "Secret Factory",
    "Wonka Gum": "Secret Factory", "Hair Toffee": "The Other Factory", "Kazookles": "The Other Factory",
}
# Approximate geographic centroids (lat, lon) - customer location proxy (dataset has no coordinates)
STATE_XY = {
 'Alabama':(32.8,-86.8),'Alberta':(55.0,-115.0),'Arizona':(34.3,-111.7),'Arkansas':(34.9,-92.4),
 'British Columbia':(53.7,-127.6),'California':(37.2,-119.5),'Colorado':(39.0,-105.5),'Connecticut':(41.6,-72.7),
 'Delaware':(39.0,-75.5),'District of Columbia':(38.9,-77.0),'Florida':(28.6,-82.4),'Georgia':(32.7,-83.4),
 'Idaho':(44.4,-114.6),'Illinois':(40.0,-89.2),'Indiana':(39.9,-86.3),'Iowa':(42.1,-93.5),'Kansas':(38.5,-98.4),
 'Kentucky':(37.5,-85.3),'Louisiana':(31.1,-92.0),'Maine':(45.4,-69.2),'Manitoba':(55.0,-97.0),
 'Maryland':(39.0,-76.8),'Massachusetts':(42.3,-71.8),'Michigan':(44.3,-85.4),'Minnesota':(46.3,-94.3),
 'Mississippi':(32.7,-89.7),'Missouri':(38.4,-92.5),'Montana':(47.0,-109.6),'Nebraska':(41.5,-99.8),
 'Nevada':(39.3,-116.6),'New Brunswick':(46.5,-66.2),'New Hampshire':(43.7,-71.6),'New Jersey':(40.1,-74.7),
 'New Mexico':(34.4,-106.1),'New York':(42.9,-75.5),'Newfoundland and Labrador':(53.1,-57.7),
 'North Carolina':(35.5,-79.4),'North Dakota':(47.4,-100.5),'Nova Scotia':(45.0,-63.0),'Ohio':(40.3,-82.8),
 'Oklahoma':(35.6,-97.5),'Ontario':(50.0,-85.0),'Oregon':(43.9,-120.6),'Pennsylvania':(40.9,-77.8),
 'Prince Edward Island':(46.4,-63.4),'Quebec':(52.0,-71.5),'Rhode Island':(41.7,-71.5),'Saskatchewan':(54.0,-106.0),
 'South Carolina':(33.9,-80.9),'South Dakota':(44.4,-100.2),'Tennessee':(35.9,-86.4),'Texas':(31.5,-99.3),
 'Utah':(39.3,-111.7),'Vermont':(44.0,-72.7),'Virginia':(37.5,-78.8),'Washington':(47.4,-120.5),
 'West Virginia':(38.6,-80.6),'Wisconsin':(44.6,-89.9),'Wyoming':(43.0,-107.6),
}

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

def normalise_lead(raw):
    """Raw (Ship-Order) gaps are 900-1640 days = three year-shifted blocks (~907/1273/1640).
    Each block's minimum is the 0-day baseline; remainder = true processing+transit days."""
    raw = np.asarray(raw)
    s = np.sort(np.unique(raw)); gaps = np.where(np.diff(s) > 100)[0]
    edges = [s[0]] + [s[i+1] for i in gaps]            # block starting points
    block = np.searchsorted(edges, raw, side='right') - 1
    return raw - np.array(edges)[block], block

def load_clean(path_or_df):
    df = pd.read_csv(path_or_df) if not isinstance(path_or_df, pd.DataFrame) else path_or_df.copy()
    df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y')
    df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y')
    df['raw_gap'] = (df['Ship Date'] - df['Order Date']).dt.days
    df['lead_days'], df['date_block'] = normalise_lead(df['raw_gap'])
    df['Product Name'] = df['Product Name'].str.strip()
    df['factory'] = df['Product Name'].map(PRODUCT_FACTORY)
    df['cust_lat'] = df['State/Province'].map(lambda s: STATE_XY[s][0])
    df['cust_lon'] = df['State/Province'].map(lambda s: STATE_XY[s][1])
    df['f_lat'] = df['factory'].map(lambda f: FACTORIES[f][0]); df['f_lon'] = df['factory'].map(lambda f: FACTORIES[f][1])
    df['distance_km'] = haversine(df.cust_lat, df.cust_lon, df.f_lat, df.f_lon)
    df['unit_price'] = df['Sales']/df['Units']; df['margin_pct'] = df['Gross Profit']/df['Sales']*100
    return df

def remove_outliers(df, col='lead_days', k=3.0):
    z = (df[col]-df[col].mean())/df[col].std()
    return df[z.abs() <= k].copy()

CAT = ['Ship Mode', 'Region', 'Division', 'Product Name']
NUM = ['distance_km', 'Units', 'unit_price']

def make_prep():
    return ColumnTransformer([('num', StandardScaler(), NUM),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])

def train_models(df, seed=42):
    """Models predict lead from product, destination, ship mode and *distance to origin factory*.
    Distance (not factory one-hot) is used so the model can score factory/product pairs never seen in history."""
    d = remove_outliers(df)
    X, y = d[CAT+NUM], d['lead_days']
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    cands = {'Linear Regression': LinearRegression(),
             'Random Forest': RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=seed),
             'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=seed)}
    fitted, rows = {}, []
    for n, m in cands.items():
        p = Pipeline([('prep', make_prep()), ('m', m)]).fit(Xtr, ytr); fitted[n] = p
        pr = p.predict(Xte)
        rows.append({'Model': n, 'RMSE': np.sqrt(mean_squared_error(yte, pr)), 'MAE': mean_absolute_error(yte, pr), 'R2': r2_score(yte, pr)})
    res = pd.DataFrame(rows).set_index('Model').round(4)
    return fitted, res, (Xte, yte)

def ensemble_predict(pipe_rf, X):
    """Mean + std across Random-Forest trees -> uncertainty for confidence score."""
    Z = pipe_rf['prep'].transform(X)
    preds = np.stack([t.predict(Z) for t in pipe_rf['m'].estimators_])
    return preds.mean(0), preds.std(0)

def simulate(df, model_pipe, rf_pipe, freight_rate=0.05, product=None, region=None, ship_mode=None):
    """For every product x candidate factory: re-predict lead time for the product's historical orders
    as if shipped from that factory. freight_rate = $ per unit per 1000 km (ASSUMPTION; dataset has no freight cost)."""
    d = df.copy()
    if region and region != 'All': d = d[d['Region'] == region]
    if ship_mode and ship_mode != 'All': d = d[d['Ship Mode'] == ship_mode]
    if product: d = d[d['Product Name'] == product]
    out = []
    for prod, g in d.groupby('Product Name'):
        cur = PRODUCT_FACTORY[prod]
        hist = df[df['Product Name'] == prod]['distance_km']
        for fac, (la, lo) in FACTORIES.items():
            X = g[CAT+NUM].copy()
            X['distance_km'] = haversine(g.cust_lat, g.cust_lon, la, lo)
            lead = model_pipe.predict(X); _, sd = ensemble_predict(rf_pipe, X)
            out.append({'Product Name': prod, 'Division': g['Division'].iloc[0], 'Factory': fac, 'Is Current': fac == cur,
                        'orders': len(g), 'units': g['Units'].sum(), 'gross_profit': g['Gross Profit'].sum(),
                        'avg_distance_km': X['distance_km'].mean(), 'pred_lead': lead.mean(), 'pred_sd': sd.mean(),
                        'extrapolation': float(np.mean((X['distance_km'] < hist.min()) | (X['distance_km'] > hist.max()))),
                        'freight': (X['distance_km']/1000*g['Units'].values*freight_rate).sum()})
    s = pd.DataFrame(out)
    if s.empty: return s
    base = s[s['Is Current']].set_index('Product Name')
    s['cur_lead'] = s['Product Name'].map(base['pred_lead']); s['cur_freight'] = s['Product Name'].map(base['freight'])
    s['cur_dist'] = s['Product Name'].map(base['avg_distance_km'])
    s['lead_red_pct'] = (s.cur_lead - s.pred_lead)/s.cur_lead*100
    s['dist_red_pct'] = (s.cur_dist - s.avg_distance_km)/s.cur_dist*100
    s['profit_delta'] = s.cur_freight - s.freight                      # + = savings
    s['profit_delta_pct'] = s.profit_delta/s.gross_profit*100
    s['confidence'] = (100*(1 - s.pred_sd/max(float(s.pred_sd.max()), 1e-9)*0.5)*(1 - 0.5*s.extrapolation)).clip(0, 100)
    return s

def rank(s, speed_weight=0.5):
    """speed_weight 1 = pure speed (lead-time + distance), 0 = pure profit. Min-max within product."""
    s = s.copy()
    s['speed_score'] = 0.5*s.lead_red_pct + 0.5*s.dist_red_pct
    def mm(x): return (x-x.min())/(x.max()-x.min()) if x.max() > x.min() else x*0
    s['score'] = 0.0
    for p, i in s.groupby('Product Name').groups.items():
        s.loc[i, 'score'] = speed_weight*mm(s.loc[i, 'speed_score']) + (1-speed_weight)*mm(s.loc[i, 'profit_delta'])
    s['risk'] = np.where((s.profit_delta < 0) | (s.confidence < 50), 'High',
                 np.where((s.confidence < 70) | (s.extrapolation > .3), 'Medium', 'Low'))
    return s.sort_values(['Product Name', 'score'], ascending=[True, False])

def cluster_routes(df, k=4, seed=42):
    r = df.groupby(['factory', 'State/Province']).agg(orders=('Row ID', 'count'), lead=('lead_days', 'mean'),
        dist=('distance_km', 'mean'), profit=('Gross Profit', 'sum')).reset_index()
    r = r[r.orders >= 5].copy()
    Z = StandardScaler().fit_transform(r[['lead', 'dist', 'orders']])
    r['cluster'] = KMeans(k, n_init=10, random_state=seed).fit_predict(Z)
    return r

def kpis(rec):
    top = rec[~rec['Is Current']].sort_values('score', ascending=False).groupby('Product Name').head(1)
    good = top[(top.lead_red_pct > 0) | (top.profit_delta > 0)]
    return {'Avg Lead Time Reduction (%)': round(top.lead_red_pct.mean(), 2),
            'Profit Impact Stability (std of %)': round(top.profit_delta_pct.std(ddof=0), 2),
            'Scenario Confidence Score': round(top.confidence.mean(), 1),
            'Recommendation Coverage (%)': round(100*len(good)/max(top['Product Name'].nunique(), 1), 1)}

In [ ]:
import pandas as pd, numpy as np, plotly.express as px, core
pd.set_option('display.width',200)
df = core.load_clean('Nassau_Candy_Distributor.csv')
print(df.shape); df.isna().sum().sum(), df.duplicated().sum()

## 2. Data preparation – important finding
Raw `Ship Date − Order Date` is **900–1,640 days**. The values form three tight blocks (~907, ~1,273, ~1,640) – the dates are year-shifted. Subtracting each block's minimum leaves the real **0–11 day** lead time (Same Day ≈ 0.4 d, Standard ≈ 5.3 d). **State this assumption in your paper and confirm it with your mentor.**

In [ ]:
print(df.raw_gap.describe())
px.histogram(df, x='raw_gap', nbins=80, title='Raw Ship−Order gap (days): three year-shifted blocks').show()
px.histogram(df, x='lead_days', color='Ship Mode', title='Normalised lead time (days)').show()

## 3. EDA

In [ ]:
print(df.groupby('Ship Mode').lead_days.agg(['count','mean','std']).round(2))
print(df.groupby('factory').agg(orders=('Row ID','count'), gross_profit=('Gross Profit','sum'), avg_dist_km=('distance_km','mean'), lead=('lead_days','mean')).round(2))
print('corr(distance, lead) =', round(df[['distance_km','lead_days']].corr().iloc[0,1],4))
px.box(df, x='Region', y='lead_days', color='Ship Mode').show()
px.scatter(df.sample(3000, random_state=1), x='distance_km', y='lead_days', color='Ship Mode', opacity=.35).show()

**Key insights so far**
- Ship mode explains almost all lead-time variation; distance/region/factory barely matter (corr ≈ 0.01).
- Two factories (Lot's O' Nuts, Wicked Choccy's) make ~97% of orders; Sugar Shack and The Other Factory are tiny.
- Each product has exactly one factory in history → factory can't be a model feature; **distance to factory** is used instead.

## 4. Predictive models (Linear / Random Forest / Gradient Boosting)

In [ ]:
models, metrics, (Xte, yte) = core.train_models(df)
metrics

## 5. Route clustering

In [ ]:
rc = core.cluster_routes(df)
print(rc.groupby('cluster').agg(routes=('orders','count'), orders=('orders','sum'), lead=('lead','mean'), dist=('dist','mean')).round(1))
px.scatter(rc, x='dist', y='lead', size='orders', color=rc.cluster.astype(str), hover_name='State/Province').show()

## 6. Scenario simulation + recommendation engine
Freight cost is **not** in the data – profit impact uses an adjustable assumption (`freight_rate`, $ per unit per 1,000 km).

In [ ]:
sim = core.simulate(df, models['Gradient Boosting'], models['Random Forest'], freight_rate=0.05)
rec = core.rank(sim, speed_weight=0.5)
top = rec[~rec['Is Current']].groupby('Product Name').head(1)
top[['Product Name','Factory','dist_red_pct','lead_red_pct','profit_delta','confidence','risk']].round(2)

In [ ]:
core.kpis(rec)

## 7. Launch the Streamlit app from Colab

In [ ]:
%%writefile app.py
import os
import streamlit as st, pandas as pd, numpy as np, plotly.graph_objects as go, plotly.express as px
import core

HERE = os.path.dirname(os.path.abspath(__file__))
DATA = next((p for p in (os.path.join(HERE, "Nassau_Candy_Distributor.csv"), os.path.join(HERE, "Nassau_Candy_Distributor.csv")) if os.path.exists(p)), None)

st.set_page_config(page_title="Nassau Candy | Factory Optimizer", page_icon="🍬", layout="wide")
PINK, GOLD, TEAL, VIO = "#FF4F9A", "#FFC83D", "#2EE6C5", "#8B5CF6"
st.markdown(f"""<style>
.hero{{padding:22px 28px;border-radius:18px;background:linear-gradient(120deg,#3A1670,#C2185B 70%,#FF8A00);margin-bottom:14px}}
.hero h1{{margin:0;font-size:2rem}} .hero p{{margin:4px 0 0;opacity:.9}}
.card{{background:#1D1233;border:1px solid #2F2150;border-radius:14px;padding:14px 16px}}
.big{{font-size:1.9rem;font-weight:800}} .lbl{{opacity:.7;font-size:.8rem;letter-spacing:.06em;text-transform:uppercase}}
.pill{{padding:2px 10px;border-radius:99px;font-size:.75rem;font-weight:700}}
.Low{{background:#12372F;color:{TEAL}}} .Medium{{background:#3B3112;color:{GOLD}}} .High{{background:#42142B;color:{PINK}}}
</style>""", unsafe_allow_html=True)

def card(label, value, sub=""):
    st.markdown(f'<div class="card"><div class="lbl">{label}</div><div class="big">{value}</div><div style="opacity:.65;font-size:.8rem">{sub}</div></div>', unsafe_allow_html=True)

# ---------- data + models ----------
@st.cache_data(show_spinner="Loading data...")
def load(file):
    return core.load_clean(file)

@st.cache_resource(show_spinner="Training models...")
def fit(_df, key):
    return core.train_models(_df)

if DATA is None:
    st.error("Data file not found. Keep Nassau_Candy_Distributor.csv in the same folder as app.py."); st.stop()
df = load(DATA)
models, metrics, (Xte, yte) = fit(df, len(df))

with st.sidebar:
    st.markdown("## 🍬 Control Room")
    products = sorted(df['Product Name'].unique())
    product = st.selectbox("Product", products, index=products.index("Wonka Bar - Milk Chocolate") if "Wonka Bar - Milk Chocolate" in products else 0)
    region = st.selectbox("Destination region", ['All'] + sorted(df['Region'].unique()))
    mode = st.selectbox("Ship mode", ['All'] + sorted(df['Ship Mode'].unique()))
    w = st.slider("Optimization priority", 0, 100, 50, format="%d%% speed", help="0 = maximise profit, 100 = maximise speed") / 100
    st.caption(f"⚡ Speed {int(w*100)}%  |  💰 Profit {int((1-w)*100)}%")
    model_name = st.selectbox("Prediction model", list(models), index=2)
    topn = st.slider("Top-N recommendations", 1, 4, 3)
    freight = st.number_input("Freight cost ($ per unit per 1,000 km)", 0.0, 2.0, 0.05, 0.01,
        help="ASSUMPTION - the dataset has no freight cost. Profit impact = change in this modelled freight cost.")

sim = core.simulate(df, models[model_name], models['Random Forest'], freight, region=region, ship_mode=mode)
rec = core.rank(sim, w)
if rec.empty: st.warning("No orders match these filters."); st.stop()

st.markdown('<div class="hero"><h1>🍬 Nassau Candy · Factory Reallocation Optimizer</h1><p>Predict. Simulate. Reassign. Balance speed against profit before you move a single product.</p></div>', unsafe_allow_html=True)

K = core.kpis(rec)
c = st.columns(4)
with c[0]: card("Lead-time reduction", f"{K['Avg Lead Time Reduction (%)']}%", "avg of best move per product")
with c[1]: card("Profit impact stability", f"±{K['Profit Impact Stability (std of %)']}%", "std of profit change")
with c[2]: card("Scenario confidence", f"{K['Scenario Confidence Score']}/100", "model certainty + data support")
with c[3]: card("Recommendation coverage", f"{K['Recommendation Coverage (%)']}%", "products with a beneficial move")
st.write("")

t1, t2, t3, t4, t5 = st.tabs(["🏭 Simulator", "🔀 What-If", "🏆 Recommendations", "🚨 Risk & Impact", "🔬 Data & Models"])

# ---------- 1 simulator ----------
with t1:
    p = rec[rec['Product Name'] == product].sort_values('pred_lead')
    cur = core.PRODUCT_FACTORY[product]
    st.subheader(f"{product}  ·  currently made at {cur}")
    a, b = st.columns([3, 2])
    with a:
        fig = go.Figure()
        fig.add_trace(go.Scattergeo(lat=[v[0] for v in core.FACTORIES.values()], lon=[v[1] for v in core.FACTORIES.values()],
            text=list(core.FACTORIES), mode='markers+text', textposition='top center',
            marker=dict(size=14, color=[PINK if f == cur else VIO for f in core.FACTORIES], line=dict(width=2, color='white'))))
        best = rec[(rec['Product Name'] == product) & (~rec['Is Current'])].iloc[0]
        la, lo = core.FACTORIES[cur]; lb, lob = core.FACTORIES[best.Factory]
        fig.add_trace(go.Scattergeo(lat=[la, lb], lon=[lo, lob], mode='lines', line=dict(width=3, color=GOLD, dash='dot')))
        fig.update_geos(scope='north america', bgcolor='rgba(0,0,0,0)', landcolor='#2A1B47', showcountries=True, countrycolor='#4B3A75', showsubunits=True, subunitcolor='#3A2A5E')
        fig.update_layout(height=430, margin=dict(l=0, r=0, t=0, b=0), paper_bgcolor='rgba(0,0,0,0)', showlegend=False)
        st.plotly_chart(fig, use_container_width=True)
        st.caption(f"Pink = current factory · gold line = best alternative at your priority ({best.Factory})")
    with b:
        fig = px.bar(p, x='pred_lead', y='Factory', orientation='h', color='Is Current', color_discrete_map={True: PINK, False: VIO},
                     labels={'pred_lead': 'Predicted lead time (days)'})
        fig.update_layout(height=430, showlegend=False, paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', margin=dict(l=0, r=0, t=10, b=0))
        fig.update_xaxes(range=[max(0, p.pred_lead.min()-0.3), p.pred_lead.max()+0.3])
        st.plotly_chart(fig, use_container_width=True)
    show = p[['Factory', 'Is Current', 'avg_distance_km', 'pred_lead', 'lead_red_pct', 'profit_delta', 'confidence', 'risk']].round(2)
    show.columns = ['Factory', 'Current', 'Avg distance (km)', 'Pred. lead (d)', 'Lead Δ%', 'Freight saving ($)', 'Confidence', 'Risk']
    st.dataframe(show, hide_index=True, use_container_width=True)

# ---------- 2 what-if ----------
with t2:
    st.subheader("Current vs recommended assignment")
    alt = rec[(rec['Product Name'] == product) & (~rec['Is Current'])]
    choice = st.selectbox("Move to factory", ["★ Best recommendation"] + list(alt['Factory']))
    row = alt.iloc[0] if choice.startswith("★") else alt[alt.Factory == choice].iloc[0]
    now = rec[(rec['Product Name'] == product) & (rec['Is Current'])].iloc[0]
    m = st.columns(3)
    m[0].metric("Predicted lead time", f"{row.pred_lead:.2f} d", f"{row.pred_lead-now.pred_lead:+.3f} d", delta_color="inverse")
    m[1].metric("Avg shipping distance", f"{row.avg_distance_km:,.0f} km", f"{row.avg_distance_km-now.avg_distance_km:+,.0f} km", delta_color="inverse")
    m[2].metric("Freight cost", f"${row.freight:,.0f}", f"{row.freight-now.freight:+,.0f}", delta_color="inverse")
    cmp_ = pd.DataFrame({'Scenario': ['Current']*2 + [f'→ {row.Factory}']*2, 'Metric': ['Lead time (days)', 'Distance (×1000 km)']*2,
                         'Value': [now.pred_lead, now.avg_distance_km/1000, row.pred_lead, row.avg_distance_km/1000]})
    fig = px.bar(cmp_, x='Metric', y='Value', color='Scenario', barmode='group', color_discrete_sequence=[PINK, TEAL])
    fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', height=340)
    st.plotly_chart(fig, use_container_width=True)
    st.info(f"Moving **{product}** from **{cur}** to **{row.Factory}** would change average shipping distance by "
            f"**{-row.dist_red_pct:+.1f}%** and predicted lead time by **{-row.lead_red_pct:+.2f}%**, "
            f"with a modelled freight impact of **${row.profit_delta:+,.0f}** on the selected orders.")

# ---------- 3 recommendations ----------
with t3:
    st.subheader(f"Top-{topn} reassignment suggestions  ·  priority: ⚡{int(w*100)}% speed / 💰{int((1-w)*100)}% profit")
    best_per = rec[~rec['Is Current']].groupby('Product Name').head(topn)
    best_per = best_per[(best_per.lead_red_pct > 0) | (best_per.profit_delta > 0)]
    tbl = best_per.sort_values('score', ascending=False).head(25)[['Product Name', 'Factory', 'avg_distance_km', 'cur_dist', 'lead_red_pct', 'dist_red_pct', 'profit_delta', 'confidence', 'risk', 'score']].round(2)
    tbl.columns = ['Product', 'Move to', 'New dist (km)', 'Current dist (km)', 'Lead Δ%', 'Distance Δ%', 'Freight saving ($)', 'Confidence', 'Risk', 'Score']
    st.dataframe(tbl, hide_index=True, use_container_width=True)
    fig = px.scatter(best_per, x='dist_red_pct', y='profit_delta', size='orders', color='risk', hover_name='Product Name',
                     hover_data=['Factory'], color_discrete_map={'Low': TEAL, 'Medium': GOLD, 'High': PINK},
                     labels={'dist_red_pct': 'Distance reduction (%)', 'profit_delta': 'Freight saving ($)'})
    fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', height=380)
    st.plotly_chart(fig, use_container_width=True)
    top1 = best_per.sort_values('score', ascending=False).iloc[0] if len(best_per) else None
    if top1 is not None:
        st.success(f"**Auto-summary:** the strongest move under your settings is **{top1['Product Name']} → {top1.Factory}** "
                   f"({top1.dist_red_pct:+.1f}% distance, ${top1.profit_delta:+,.0f} freight, {top1.confidence:.0f}/100 confidence, {top1.risk} risk). "
                   f"{K['Recommendation Coverage (%)']}% of products have at least one beneficial alternative.")

# ---------- 4 risk ----------
with t4:
    st.subheader("Profit-impact alerts & high-risk moves")
    mv = rec[~rec['Is Current']].copy()
    bad = mv[mv.profit_delta < 0].sort_values('profit_delta').head(10)
    risky = mv[mv.risk == 'High'].sort_values('confidence').head(10)
    a, b = st.columns(2)
    with a:
        st.markdown("**🔻 Moves that reduce profit**")
        for _, r in bad.iterrows():
            st.warning(f"{r['Product Name']} → {r.Factory}: freight +${-r.profit_delta:,.0f} ({r.profit_delta_pct:+.1f}% of gross profit)")
        if bad.empty: st.success("No profit-eroding moves in this selection.")
    with b:
        st.markdown("**⚠️ High-risk / low-confidence moves**")
        for _, r in risky.iterrows():
            st.error(f"{r['Product Name']} → {r.Factory}: confidence {r.confidence:.0f}/100, {r.extrapolation*100:.0f}% of orders outside historical distance range")
        if risky.empty: st.success("No high-risk moves in this selection.")
    fig = px.density_heatmap(mv, x='Factory', y='Product Name', z='profit_delta', histfunc='avg', color_continuous_scale='RdYlGn',
                             labels={'profit_delta': 'Freight saving ($)'})
    fig.update_layout(height=520, paper_bgcolor='rgba(0,0,0,0)')
    st.plotly_chart(fig, use_container_width=True)

# ---------- 5 data & models ----------
with t5:
    st.subheader("Model comparison")
    st.dataframe(metrics, use_container_width=True)
    st.caption("RMSE penalises large errors · MAE = average error (days) · R² = variance explained.")
    d = df
    a, b = st.columns(2)
    with a:
        fig = px.box(d, x='Ship Mode', y='lead_days', color='Ship Mode', color_discrete_sequence=[PINK, GOLD, TEAL, VIO], labels={'lead_days': 'Lead time (days)'})
        fig.update_layout(showlegend=False, paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)'); st.plotly_chart(fig, use_container_width=True)
    with b:
        s = d.sample(min(3000, len(d)), random_state=1)
        fig = px.scatter(s, x='distance_km', y='lead_days', opacity=.35, color='Ship Mode', trendline=None, labels={'distance_km': 'Distance factory→customer (km)', 'lead_days': 'Lead time (days)'})
        fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)'); st.plotly_chart(fig, use_container_width=True)
    corr = d[['lead_days', 'distance_km']].corr().iloc[0, 1]
    st.warning(f"**Data finding:** correlation between distance and lead time is only **{corr:.3f}**. Ship mode explains almost all lead-time variation, "
               "so predicted lead-time gains from moving factories are small and the recommendations are driven mainly by shorter distance and freight cost.")
    st.markdown("**Route clusters** (factory × destination state, KMeans)")
    rc = core.cluster_routes(d)
    fig = px.scatter(rc, x='dist', y='lead', size='orders', color=rc['cluster'].astype(str), hover_name='State/Province', hover_data=['factory'],
                     labels={'dist': 'Avg distance (km)', 'lead': 'Avg lead time (d)', 'color': 'Cluster'})
    fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)'); st.plotly_chart(fig, use_container_width=True)
    with st.expander("Data preparation notes"):
        st.markdown("- Raw Ship−Order gaps are 900–1,640 days (three year-shifted blocks). Each block's minimum is treated as 0, leaving 0–11 real days.\n"
                    "- Customer coordinates are state/province centroids (dataset has none). Factory coordinates are from the project brief.\n"
                    "- Factory is derived from the product→factory table in the brief.\n"
                    "- Outliers beyond 3σ are removed before training. Models use distance-to-factory instead of a factory label so alternative factories can be scored.")


In [ ]:
import os
os.makedirs('.streamlit', exist_ok=True)
open('.streamlit/config.toml','w').write('''[theme]
base="dark"
primaryColor="#FF4F9A"
backgroundColor="#120B1F"
secondaryBackgroundColor="#1D1233"
textColor="#F5EEFF"
''')

In [ ]:
!npm install -s localtunnel
!streamlit run app.py --server.port 8501 &>/content/log.txt &
!echo 'Tunnel password (paste on the page):' && curl -s ipv4.icanhazip.com
!npx localtunnel --port 8501